<a href="https://colab.research.google.com/github/rudraworkswcode/QMAMA/blob/main/maternal_quantum_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Step 1: Upload the csv file titled "Maternal Health Risk Data Set.csv" uploaded in the github repository into your google drive. Ensure to mount your google drive to your live colab enviroment. Connect to the normal CPU for now. Using the GPU will be specified.

In [ ]:
#mount google drive and give necessary permissions
from google.colab import drive
drive.mount('/content/drive')
#write the project path to where the maternal health risk data set.csv is uploaded. you can access it from the files, located in the left toolbar.
project_path = '/content/drive/MyDrive/{your file name}'

Step 2: Import the data into a python dataframe

In [ ]:
import pandas as pd

# Load the dataset from the current working directory (your Drive folder)
df = pd.read_csv('Maternal Health Risk Data Set.csv')

# Display structure to confirm successful load
print(df.info())
df.head()

Step 3: Perform Principal Component Analysis, in order to effectively run the Quantum Machine Learning Algorithms, and provide a 1:1 comparison to classical algorithms. Perform other necesary cleaning, and save it as a new csv file.

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA


df = pd.read_csv('Maternal Health Risk Data Set.csv')
print("--- Step 0: Original Data ---")
print(f"Shape: {df.shape}")
print(df.head(3))

# 1. Remove Duplicates
# IoT-based clinical data often contains redundant entries.
df = df.drop_duplicates()
print("\n--- Step 1: After Removing Duplicates ---")
print(f"New Shape: {df.shape}")

# 2. Label Encoding
# Converts categorical risk levels to numerical format (0, 1, 2).
le = LabelEncoder()
df['RiskLevel'] = le.fit_transform(df['RiskLevel'])
print("\n--- Step 2: After Label Encoding ---")
print(df[['RiskLevel']].head(3))
print(f"Encoded Classes: {list(le.classes_)}")

# 3. Feature-Target Split
X = df.drop('RiskLevel', axis=1)
y = df['RiskLevel']

# 4. Standardisation
# Ensures all features contribute equally to the quantum state.
# Formula: $z = \frac{x - \mu}{\sigma}$
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("\n--- Step 4: After Standardisation (First Row) ---")
print(X_scaled[0])

# 5. Principal Component Analysis (PCA)
# Reduces 6 features into 2 components for 2-qubit mapping.
pca = PCA(n_components=2)
X_reduced = pca.fit_transform(X_scaled)
print("\n--- Step 5: After PCA ---")
print(f"Reduced Shape: {X_reduced.shape}")
print("First Row Components:", X_reduced[0])

# 6. Save Processed Data for persistence
processed_df = pd.DataFrame(X_reduced, columns=['PC1', 'PC2'])
processed_df['Target'] = y.values
processed_df.to_csv('Processed_Maternal_Data.csv', index=False)
print("\nPreprocessing complete. Progress saved to Google Drive.")

Step 4: Disconnect from the CPU and connect to the GPU now. You will need to remount the drive, and again define the file paths.

In [ ]:
import pandas as pd
from google.colab import drive

# 2. Define the file path
file_path = '/content/drive/MyDrive/{your_file_name}'

# 3. Load the DataFrame
df = pd.read_csv(file_path)

# 4. Define Features (X) and Target (y)
X = df[['PC1', 'PC2']]
y = df['Target']

# 5. Data Verification
print("DataFrame successfully loaded.")
print(f"Total Samples: {len(df)}")
print(df.head())

Step 5: Perform 5 fold iterative predictions using classical prediction mechanisms, Support Vector Machines and Random Forest Classifier. Save the output to a new dedicated folder.

In [ ]:
import pandas as pd
import numpy as np
import os
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

# 1. Initialise Environment and Data
file_path = '/content/drive/MyDrive/{your_file_name}'
save_path = '/content/drive/MyDrive/{your_file_name}'


# 3. Iterative Execution (5 Runs)
print("Starting 5-fold iterative evaluation...")
for i in range(5):
    # Split with a unique random state for statistical variety
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=i, stratify=y
    )

    # Feature Standardisation
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_val_s = scaler.transform(X_val)

    # Train Classical Models
    svm = SVC(kernel='rbf', C=1.0).fit(X_train_s, y_train)
    rf = RandomForestClassifier(n_estimators=100, random_state=i).fit(X_train_s, y_train)

    # Calculate and Record Metrics
    svm_acc = accuracy_score(y_val, svm.predict(X_val_s))
    rf_acc = accuracy_score(y_val, rf.predict(X_val_s))
    svm_f1 = f1_score(y_val, svm.predict(X_val_s), average='macro')
    rf_f1 = f1_score(y_val, rf.predict(X_val_s), average='macro')

    results.append({
        'Iteration': i + 1,
        'SVM_Accuracy': svm_acc,
        'RF_Accuracy': rf_acc,
        'SVM_F1_Macro': svm_f1,
        'RF_F1_Macro': rf_f1
    })

# 4. Aggregate and Export Results
results_df = pd.DataFrame(results)
summary_stats = results_df.agg(['mean', 'std']).drop('Iteration', axis=1)

results_df.to_csv(f'{save_path}multi_run_raw_results.csv', index=False)
summary_stats.to_csv(f'{save_path}classical_benchmark_summary.csv')

print("\n--- Aggregated Performance Metrics (Averages) ---")
print(summary_stats.loc['mean'])
print("\n--- Statistical Variation (Std Dev) ---")
print(summary_stats.loc['std'])
print(f"\nFiles successfully saved to: {save_path}")

Step 6: Install necessary Quantum Computer Simulating libraries from IBM

In [ ]:
!pip install qiskit qiskit-machine-learning qiskit-aer

Step 7: Running Quantum Machine Learning for 2 PCA, and 2 Qubits. Saving the running every few cycles, to ensure we don't lose progress.

In [ ]:
#452 QUANTUM 2 PCA 2 QUBIT
import pandas as pd
import numpy as np
import os
import time
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score

# Quantum-specific imports
from qiskit.circuit.library import ZZFeatureMap
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import SamplerV2 as AerSampler
from qiskit_machine_learning.state_fidelities import ComputeUncompute
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC

# 1. Environment Setup
drive.mount('/content/drive')
save_path = '/content/drive/MyDrive/{your_file_name}'
if not os.path.exists(save_path):
    os.makedirs(save_path)

# 2. Initialise Quantum Components
backend = AerSimulator(max_parallel_experiments=0, max_parallel_threads=0)
sampler = AerSampler.from_backend(backend)
fidelity = ComputeUncompute(sampler=sampler)
feature_map = ZZFeatureMap(feature_dimension=2, reps=2, entanglement='linear').decompose()
q_kernel = FidelityQuantumKernel(feature_map=feature_map, fidelity=fidelity)

quantum_results = []
checkpoint_file = f'{save_path}quantum_checkpoint.csv'

# 3. Iterative Evaluation with Automated Saving
print("Commencing iterative Quantum evaluation with automated checkpointing...")

for i in range(5):
    # Data Splitting and Scaling
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=i, stratify=y
    )
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_val_s = scaler.transform(X_val)

    # Execution
    start_time = time.time()
    qsvc = QSVC(quantum_kernel=q_kernel)
    qsvc.fit(X_train_s, y_train)
    preds = qsvc.predict(X_val_s)

    # Metrics Calculation
    acc = accuracy_score(y_val, preds)
    f1 = f1_score(y_val, preds, average='macro')

    # Record Progress
    iteration_data = {
        'Iteration': i + 1,
        'Quantum_Accuracy': acc,
        'Quantum_F1_Macro': f1,
        'Timestamp': time.strftime("%Y-%m-%d %H:%M:%S")
    }
    quantum_results.append(iteration_data)

    # AUTOMATED SAVE: Export current progress after every iteration
    # This acts as a checkpoint if the runtime disconnects
    checkpoint_df = pd.DataFrame(quantum_results)
    checkpoint_df.to_csv(checkpoint_file, index=False)

    duration = (time.time() - start_time) / 60
    print(f"Iteration {i+1} complete ({duration:.2f} mins). Accuracy: {acc:.4f}. Progress saved to Drive.")

# 4. Final Aggregation
summary_df = checkpoint_df.agg(['mean', 'std']).drop(['Iteration', 'Timestamp'], axis=1)
summary_df.to_csv(f'{save_path}quantum_statistical_summary.csv')

print(f"\nFinal benchmark data successfully saved to: {save_path}")

Step 8: Perform 5 fold iterative predictions using classical prediction mechanisms, Support Vector Machines and Random Forest Classifier. Truncating the dataset to 200.

In [ ]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from google.colab import drive

# 1. Environment Setup
drive.mount('/content/drive')
raw_path = '/content/drive/MyDrive/{your_file_name}'
proc_path = '/content/drive/MyDrive/{your_file_name}'
save_path_results = "/content/drive/MyDrive/{your_file_name}"

# 2. Data Cleaning and Merging
df_raw = pd.read_csv(raw_path)
df_proc = pd.read_csv(proc_path)
df_raw['Target'] = df_proc['Target']
df_clean = df_raw.drop_duplicates().dropna()

X_raw = df_clean.drop(columns=['Target']).select_dtypes(include=[np.number])
y_raw = df_clean['Target']

# 3. Stratified Truncation (200 Samples)
_, X_trunc, _, y_trunc = train_test_split(
    X_raw, y_raw, test_size=200, random_state=42, stratify=y_raw
)

# 4. PCA (4 Components) and Angular Scaling (0 to pi)
pca = PCA(n_components=4)
X_pca = pca.fit_transform(X_trunc)
scaler = MinMaxScaler(feature_range=(0, np.pi))
X_scaled = scaler.fit_transform(X_pca)

# 5. Iterative Evaluation Loop
classical_results = []

print("Commencing Classical Baseline Evaluation (5 Runs)...")

for i in range(5):
    X_train, X_val, y_train, y_val = train_test_split(
        X_scaled, y_trunc, test_size=0.2, random_state=i, stratify=y_trunc
    )

    # Support Vector Machine (Optimised with C=100.0 to match QSVC)
    svm = SVC(kernel='rbf', C=100.0)
    svm.fit(X_train, y_train)
    svm_preds = svm.predict(X_val)

    # Random Forest Classifier
    rf = RandomForestClassifier(n_estimators=100, random_state=i)
    rf.fit(X_train, y_train)
    rf_preds = rf.predict(X_val)

    classical_results.append({
        'Iteration': i + 1,
        'SVM_Accuracy': accuracy_score(y_val, svm_preds),
        'SVM_F1_Macro': f1_score(y_val, svm_preds, average='macro'),
        'RF_Accuracy': accuracy_score(y_val, rf_preds),
        'RF_F1_Macro': f1_score(y_val, rf_preds, average='macro')
    })

    print(f"Run {i+1} complete.")

# 6. Final Summary and Export
results_df = pd.DataFrame(classical_results)
summary = results_df.agg(['mean', 'std']).drop('Iteration', axis=1)

print("\n--- Classical Baseline Metrics (200 Samples) ---")
print(summary.loc['mean'])

if save_path_results:
    results_df.to_csv(save_path_results, index=False)

Step 9: Running Quantum Machine Learning for 4 PCA, and 4 Qubits, 200 samples. Saving the running every few cycles, to ensure we don't lose progress.

In [ ]:
import pandas as pd
import numpy as np
import os
import time
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score
from google.colab import drive

# Quantum-specific imports
from qiskit.circuit.library import zz_feature_map
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import SamplerV2 as AerSampler
from qiskit_machine_learning.state_fidelities import ComputeUncompute
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC

# 1. Environment Setup
drive.mount('/content/drive')
save_path = '/content/drive/MyDrive/{your_file_name}'

# 1. Data Merging and Cleaning
# Ensure the Target from your processed file is mapped to the raw data
df_raw = pd.read_csv('/content/drive/MyDrive/{your_file_name}')
df_proc = pd.read_csv('/content/drive/MyDrive/{your_file_name}')
df_raw['Target'] = df_proc['Target']
df_clean = df_raw.drop_duplicates().dropna()

X_raw = df_clean.drop(columns=['Target']).select_dtypes(include=[np.number])
y_raw = df_clean['Target']

# 2. Stratified Truncation (200 Samples)
_, X_trunc, _, y_trunc = train_test_split(
    X_raw, y_raw, test_size=200, random_state=42, stratify=y_raw
)

# 3. PCA (4 Components) and Angular Scaling
pca = PCA(n_components=4)
X_pca = pca.fit_transform(X_trunc)
scaler = MinMaxScaler(feature_range=(0, np.pi))
X_scaled = scaler.fit_transform(X_pca)

# 4. Quantum Component Configuration
backend = AerSimulator(max_parallel_experiments=0, max_parallel_threads=0)
sampler = AerSampler.from_backend(backend)
fidelity = ComputeUncompute(sampler=sampler)

# 4-Qubit Map with Circular Entanglement
feature_map = zz_feature_map(feature_dimension=4, reps=2, entanglement='circular').decompose()
q_kernel = FidelityQuantumKernel(feature_map=feature_map, fidelity=fidelity)

# 5. Iterative Execution
quantum_results = []
save_path = '/content/drive/MyDrive/{your_file_name}'
checkpoint_file = os.path.join(save_path, 'quantum_optimised_200_checkpoint.csv')

print("Starting Optimised 4-Qubit QSVC Simulation...")

for i in range(5):
    X_train, X_val, y_train, y_val = train_test_split(
        X_scaled, y_trunc, test_size=0.2, random_state=i, stratify=y_trunc
    )

    start_time = time.time()

    # Using high C-value to sharpen decision boundaries
    qsvc = QSVC(quantum_kernel=q_kernel, C=100.0)
    qsvc.fit(X_train, y_train)

    preds = qsvc.predict(X_val)
    acc = accuracy_score(y_val, preds)
    f1 = f1_score(y_val, preds, average='macro')

    iteration_data = {
        'Iteration': i + 1,
        'Accuracy': acc,
        'F1_Macro': f1,
        'Duration_Mins': (time.time() - start_time) / 60
    }
    quantum_results.append(iteration_data)
    pd.DataFrame(quantum_results).to_csv(checkpoint_file, index=False)

    print(f"Run {i+1}: Acc={acc:.4f} | F1={f1:.4f} | Time={iteration_data['Duration_Mins']:.2f}m")

# 6. Final Summary
summary = pd.DataFrame(quantum_results).agg(['mean', 'std']).drop('Iteration', axis=1)
summary.to_csv(os.path.join(save_path, 'quantum_optimised_final_summary.csv'))
print("\n--- Optimised Quantum Performance ---")
print(summary.loc['mean'])

Step 10: Perform 5 fold iterative predictions using classical prediction mechanisms, Support Vector Machines and Random Forest Classifier. Truncating the dataset to 150.

In [ ]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

# 1. Data Loading and Cleaning (Matching your Quantum Setup)
raw_path = '/content/drive/MyDrive/{your_file_name}'
processed_path = '/content/drive/MyDrive/{your_file_name}'
save_path_svm = "/content/drive/MyDrive/{your_file_name}"
save_path_rf = "/content/drive/MyDrive/{your_file_name}"

df_raw = pd.read_csv(raw_path)
df_processed = pd.read_csv(processed_path)

df_raw['Target'] = df_processed['Target']
df_cleaned = df_raw.drop_duplicates().dropna()
df_final = pd.get_dummies(df_cleaned)

X_all = df_final.drop(columns=['Target'])
y_all = df_final['Target']

# 2. Stratified Truncation to 150 Samples
_, X_trunc, _, y_trunc = train_test_split(
    X_all, y_all, test_size=150, random_state=42, stratify=y_all
)

# 3. PCA (4 Components) and Scaling (Identical to Quantum Run)
pca = PCA(n_components=4)
X_pca = pca.fit_transform(X_trunc)
scaler = MinMaxScaler(feature_range=(0, np.pi))
X_scaled = scaler.fit_transform(X_pca)

# 4. Iterative Evaluation Loop
classical_results = []

print("Commencing Classical Baseline Evaluation (5 Runs)...")
for i in range(5):
    X_train, X_val, y_train, y_val = train_test_split(
        X_scaled, y_trunc, test_size=0.2, random_state=i, stratify=y_trunc
    )

    # --- Support Vector Machine ---
    svm = SVC(kernel='rbf', C=10.0)
    svm.fit(X_train, y_train)
    svm_preds = svm.predict(X_val)

    # --- Random Forest ---
    rf = RandomForestClassifier(n_estimators=100, random_state=i)
    rf.fit(X_train, y_train)
    rf_preds = rf.predict(X_val)

    # Record Metrics
    classical_results.append({
        'Iteration': i + 1,
        'SVM_Accuracy': accuracy_score(y_val, svm_preds),
        'SVM_F1': f1_score(y_val, svm_preds, average='macro'),
        'RF_Accuracy': accuracy_score(y_val, rf_preds),
        'RF_F1': f1_score(y_val, rf_preds, average='macro')
    })

    print(f"Run {i+1} complete.")

# 5. Summary Statistics
results_df = pd.DataFrame(classical_results)
summary = results_df.agg(['mean', 'std']).drop('Iteration', axis=1)

print("\n--- Classical Baseline Metrics ---")
print(summary.loc['mean'])

# 6. Save Results
if save_path_svm:
    results_df.to_csv(save_path_svm, index=False)
if save_path_rf:
    summary.to_csv(save_path_rf)

Step 11: Running Quantum Machine Learning for 4 PCA, and 4 Qubits, 150 samples. Saving the running every few cycles, to ensure we don't lose progress.

In [ ]:
import pandas as pd
import numpy as np
import os
import time
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score

# Quantum-specific imports
from qiskit.circuit.library import zz_feature_map
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import SamplerV2 as AerSampler
from qiskit_machine_learning.state_fidelities import ComputeUncompute
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC

# 1. Initialise Environment
drive.mount('/content/drive')
raw_path = '/content/drive/MyDrive/{your_file_name}'
processed_path = '/content/drive/MyDrive/{your_file_name}'
save_path = '/content/drive/MyDrive/{your_file_name}'

if not os.path.exists(save_path):
    os.makedirs(save_path)

# 2. Data Cleaning and Merging
df_raw = pd.read_csv(raw_path)
df_processed = pd.read_csv(processed_path)

# Merge Target from processed data (assuming index alignment)
df_raw['Target'] = df_processed['Target']

# Remove duplicates and handle missing values
df_cleaned = df_raw.drop_duplicates().dropna()

# Categorical conversion (One-Hot Encoding for non-numeric features)
df_final = pd.get_dummies(df_cleaned)

X_all = df_final.drop(columns=['Target'])
y_all = df_final['Target']

# 3. Stratified Truncation to 150 Samples
_, X_trunc, _, y_trunc = train_test_split(
    X_all, y_all, test_size=150, random_state=42, stratify=y_all
)

# 4. Dimensionality Reduction (4 Principal Components)
pca = PCA(n_components=4)
X_pca = pca.fit_transform(X_trunc)

# Angular scaling for Quantum gates (0 to pi)
scaler = MinMaxScaler(feature_range=(0, np.pi))
X_scaled = scaler.fit_transform(X_pca)

print(f"Preprocessing complete. {len(X_scaled)} samples retained.")
print(f"Cumulative Variance (4 PCs): {sum(pca.explained_variance_ratio_):.4f}")

# 5. Quantum Configuration
backend = AerSimulator(max_parallel_experiments=0, max_parallel_threads=0)
sampler = AerSampler.from_backend(backend)
fidelity = ComputeUncompute(sampler=sampler)

# 4-Qubit Map with reps=3 and circular entanglement
feature_map = zz_feature_map(feature_dimension=4, reps=3, entanglement='circular').decompose()
q_kernel = FidelityQuantumKernel(feature_map=feature_map, fidelity=fidelity)

# 6. Iterative Execution (5 Runs)
quantum_results = []
checkpoint_file = os.path.join(save_path, 'quantum_150_sample_checkpoint.csv')

print("Starting Quantum Simulation (Estimated duration: 45-60 minutes)...")
for i in range(5):
    X_train, X_val, y_train, y_val = train_test_split(
        X_scaled, y_trunc, test_size=0.2, random_state=i, stratify=y_trunc
    )

    start_time = time.time()

    # Train QSVC with C=10.0 optimisation
    qsvc = QSVC(quantum_kernel=q_kernel, C=10.0)
    qsvc.fit(X_train, y_train)

    preds = qsvc.predict(X_val)
    acc = accuracy_score(y_val, preds)
    f1 = f1_score(y_val, preds, average='macro')

    iteration_data = {
        'Iteration': i + 1,
        'Accuracy': acc,
        'F1_Macro': f1,
        'Time_Mins': (time.time() - start_time) / 60
    }
    quantum_results.append(iteration_data)
    pd.DataFrame(quantum_results).to_csv(checkpoint_file, index=False)

    print(f"Run {i+1}: Acc={acc:.4f} | Time={iteration_data['Time_Mins']:.2f}m")

# 7. Final Statistical Summary
summary = pd.DataFrame(quantum_results).agg(['mean', 'std']).drop('Iteration', axis=1)
summary.to_csv(os.path.join(save_path, 'quantum_final_150_summary.csv'))

print("\n--- Final Research Metrics ---")
print(summary.loc['mean'])

Step 12: Perform 5 fold iterative predictions using classical prediction mechanisms, Support Vector Machines and Random Forest Classifier. Truncating the dataset to 100.

In [ ]:
import pandas as pd
import numpy as np
import os
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

# 1. Environment Setup
file_path = '/content/drive/MyDrive/{your_file_name}'
save_path = '/content/drive/MyDrive/{your_file_name}'

# 2. Data Loading and 100-Sample Truncation
df = pd.read_csv(file_path)

# Stratified truncation to maintain class balance (Identical to Quantum setup)
_, df_truncated = train_test_split(
    df,
    test_size=100,
    random_state=42,
    stratify=df['Target']
)

X_full = df_truncated[['PC1', 'PC2']]
y_full = df_truncated['Target']

# 3. Iterative Evaluation Loop
classical_results = []

print("Commencing 5-run classical baseline evaluation...")

for i in range(5):
    # Split for the current iteration
    X_train, X_val, y_train, y_val = train_test_split(
        X_full, y_full, test_size=0.2, random_state=i, stratify=y_full
    )

    # Standardisation
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_val_s = scaler.transform(X_val)

    # --- Support Vector Machine (RBF Kernel) ---
    svm = SVC(kernel='rbf', C=1.0, random_state=42)
    svm.fit(X_train_s, y_train)
    svm_preds = svm.predict(X_val_s)

    # --- Random Forest ---
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X_train_s, y_train)
    rf_preds = rf.predict(X_val_s)

    # Metrics Calculation
    classical_results.append({
        'Iteration': i + 1,
        'SVM_Accuracy': accuracy_score(y_val, svm_preds),
        'SVM_F1_Macro': f1_score(y_val, svm_preds, average='macro'),
        'RF_Accuracy': accuracy_score(y_val, rf_preds),
        'RF_F1_Macro': f1_score(y_val, rf_preds, average='macro')
    })

    print(f"Iteration {i+1} complete.")

# 4. Statistical Summary
results_df = pd.DataFrame(classical_results)
summary_df = results_df.agg(['mean', 'std']).drop('Iteration', axis=1)

# Export results
results_df.to_csv(os.path.join(save_path, 'classical_baseline_raw.csv'), index=False)
summary_df.to_csv(os.path.join(save_path, 'classical_baseline_summary.csv'))

print("\n--- Final Classical Baseline Summary ---")
print(summary_df.loc['mean'])

Step 13: Step 11: Running Quantum Machine Learning for 4 PCA, and 4 Qubits, 100 samples. Saving the running every few cycles, to ensure we don't lose progress.

In [ ]:
import pandas as pd
import numpy as np
import os
import time
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score

# Quantum-specific imports
from qiskit.circuit.library import zz_feature_map
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import SamplerV2 as AerSampler
from qiskit_machine_learning.state_fidelities import ComputeUncompute
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC

# 1. Environment and Path Setup
drive.mount('/content/drive')
raw_path = '/content/drive/MyDrive/{your_file_name}'
processed_path = '/content/drive/MyDrive/{your_file_name}'
save_path = '/content/drive/MyDrive/{your_file_name}'

# 2. Data Cleaning and Merging
df_raw = pd.read_csv(raw_path)
df_processed = pd.read_csv(processed_path)

# Merge Target from processed data into the raw dataset
df_raw['Target'] = df_processed['Target']

# Cleaning: Remove duplicates and handle missing values
df_cleaned = df_raw.drop_duplicates().dropna()

# Categorical conversion (One-Hot Encoding for any non-numeric features in raw data)
df_final = pd.get_dummies(df_cleaned)

X_all = df_final.drop(columns=['Target'])
y_all = df_final['Target']

# 3. Stratified Truncation (200 Samples recommended for 4-PCA complexity)
_, X_trunc, _, y_trunc = train_test_split(
    X_all, y_all, test_size=200, random_state=42, stratify=y_all
)

# 4. Dimensionality Reduction (4 Principal Components)
pca = PCA(n_components=4)
X_pca = pca.fit_transform(X_trunc)

# Angular scaling for Quantum gates (0 to pi)
scaler = MinMaxScaler(feature_range=(0, np.pi))
X_scaled = scaler.fit_transform(X_pca)

print(f"Preprocessing complete. PCA-4 captures {sum(pca.explained_variance_ratio_):.2%} of variance.")

# 5. Quantum Configuration
backend = AerSimulator(max_parallel_experiments=0, max_parallel_threads=0)
sampler = AerSampler.from_backend(backend)
fidelity = ComputeUncompute(sampler=sampler)

# 4-Qubit Map with reps=2 and circular entanglement
feature_map = zz_feature_map(feature_dimension=4, reps=2, entanglement='circular').decompose()
q_kernel = FidelityQuantumKernel(feature_map=feature_map, fidelity=fidelity)

# 6. Iterative Execution (5 Runs)
quantum_results = []
checkpoint_file = os.path.join(save_path, 'quantum_4pca_4qubit_checkpoint.csv')

print("Starting 4-Qubit 4-PCA QSVC Simulation...")
for i in range(5):
    X_train, X_val, y_train, y_val = train_test_split(
        X_scaled, y_trunc, test_size=0.2, random_state=i, stratify=y_trunc
    )

    start_time = time.time()

    # Train QSVC with C=10.0 for stricter boundary definition
    qsvc = QSVC(quantum_kernel=q_kernel, C=10.0)
    qsvc.fit(X_train, y_train)

    # Prediction and Metrics
    preds = qsvc.predict(X_val)
    acc = accuracy_score(y_val, preds)
    f1 = f1_score(y_val, preds, average='macro')

    iteration_data = {
        'Iteration': i + 1,
        'Accuracy': acc,
        'F1_Macro': f1,
        'Time_Mins': (time.time() - start_time) / 60
    }
    quantum_results.append(iteration_data)
    pd.DataFrame(quantum_results).to_csv(checkpoint_file, index=False)

    print(f"Run {i+1}: Acc={acc:.4f} | F1={f1:.4f} | Time={iteration_data['Time_Mins']:.2f}m")

# 7. Final Statistical Summary
summary = pd.DataFrame(quantum_results).agg(['mean', 'std']).drop('Iteration', axis=1)
summary.to_csv(os.path.join(save_path, 'quantum_4pca_4qubit_summary.csv'))

print("\n--- Final Research Metrics (4-PCA/4-Qubit) ---")
print(summary.loc['mean'])

Step 14:  Tabulating and graphing findings, to improve understanding of obtained data. Sample data used, to simulate how graphs can be made. Multiple code snippets for various types of graph given.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Data derived from experimental results
samples = ['N=452', 'N=200', 'N=150']
qsvc_acc = [0.5165, 0.3563, 0.4333]
svm_acc = [0.612, 0.3950, 0.4012]
rfc_acc = [0.5956, 0.4000, 0.38]

qsvc_f1 = [0.2771, 0.32225, 0.3826]
svm_f1 = [0.4678, 0.3226, 0.3323]
rfc_f1 = [0.4923, 0.3012, 0.3240]

# User-defined colour palette
DARK_GREEN = '#006CC4'  # Medium Blue
LIGHT_GREEN = '#F7BC4F' # Light Blue
ACCENT_GRAY = '#f96253' # Sage Green

x = np.arange(len(samples))
width = 0.25

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Plot 1: Accuracy with black borders (edgecolor)
rects1 = ax1.bar(x - width, qsvc_acc, width, label='QSVC (Quantum)',
                 color=DARK_GREEN, edgecolor='black', linewidth=1.2)
ax1.bar(x, svm_acc, width, label='SVM (Classical)',
        color=LIGHT_GREEN, edgecolor='black', linewidth=1.2)
ax1.bar(x + width, rfc_acc, width, label='RFC (Classical)',
        color=ACCENT_GRAY, edgecolor='black', linewidth=1.2)

ax1.set_title('Model Accuracy vs Sample Size', fontsize=14, color=DARK_GREEN, fontweight='bold')
ax1.set_ylabel('Accuracy Score', color=DARK_GREEN, fontweight='bold')

# Plot 2: F1-Score with black borders (edgecolor)
rects2 = ax2.bar(x - width, qsvc_f1, width, label='QSVC (Quantum)',
                 color=DARK_GREEN, edgecolor='black', linewidth=1.2)
ax2.bar(x, svm_f1, width, label='SVM (Classical)',
        color=LIGHT_GREEN, edgecolor='black', linewidth=1.2)
ax2.bar(x + width, rfc_f1, width, label='RFC (Classical)',
        color=ACCENT_GRAY, edgecolor='black', linewidth=1.2)

ax2.set_title('F1-Score vs Sample Size', fontsize=14, color=DARK_GREEN, fontweight='bold')
ax2.set_ylabel('F1-Score', color=DARK_GREEN, fontweight='bold')

# Add asterisks to preliminary QSVC data
for rect in [rects1[0], rects1[1], rects2[0], rects2[1]]:
    height = rect.get_height()
    target_ax = ax1 if rect in rects1 else ax2
    target_ax.text(rect.get_x() + rect.get_width()/2., height, '*',
                   ha='center', va='bottom', fontsize=18, color='red', fontweight='bold')

# Formatting for solid white background and clear legibility
for ax in [ax1, ax2]:
    ax.set_xticks(x)
    ax.set_xticklabels(samples, color=DARK_GREEN, fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.patch.set_facecolor('white')
    ax.tick_params(axis='y', colors=DARK_GREEN)

fig.patch.set_facecolor('white')
fig.legend(['QSVC (Quantum)', 'SVM (Classical)', 'RFC (Classical)'],
           loc='lower center', ncol=3, frameon=False, bbox_to_anchor=(0.5, -0.05), fontsize=11)

plt.tight_layout()

# Save as high-resolution image with solid white background
plt.savefig('maternal_health_performance.png', transparent=False, facecolor='white', dpi=300, bbox_inches='tight')
plt.show()

# Load the cleaned dataset
df = pd.read_csv('/content/drive/MyDrive/{your_file_name}')

# Set visual theme
sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 6), facecolor='white')

# Define the clinical order for logical comparison
risk_order = ['low risk', 'mid risk', 'high risk']

# Generate Boxplot: Age distributed by RiskLevel
sns.boxplot(x='RiskLevel', y='Age', data=df, order=risk_order, palette='viridis', width=0.6)

# Formatting for presentation consistency
plt.title('Distribution of Maternal Age by Risk Level', fontsize=14, fontweight='bold', color='#006CC4')
plt.xlabel('Risk Level', fontweight='bold', color='#006CC4')
plt.ylabel('Age (Years)', fontweight='bold', color='#006CC4')

plt.tight_layout()
plt.savefig('age_by_risk_boxplot.png', dpi=300)
plt.show()

sns.set_theme(style="whitegrid")

# 1. Combined Boxplot of All Numerical Variables
plt.figure(figsize=(12, 6))
# Selecting only numerical columns for the combined plot
numerical_df = df.select_dtypes(include=['float64', 'int64'])
sns.boxplot(data=numerical_df, palette="husl")
plt.title('Boxplots of All Variables', fontsize=14, fontweight='bold')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('combined_boxplot.png', dpi=300)
plt.show()

# 2. Risk Level Class Distribution (Countplot)
plt.figure(figsize=(8, 5))
# Define order to match clinical progression
risk_order = ['low risk', 'mid risk', 'high risk']
sns.countplot(x='RiskLevel', data=df, order=risk_order, palette='viridis')
plt.title('Risk Level Class Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Risk Level')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('risk_distribution.png', dpi=300)
plt.show()